In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
markers_df = pd.read_csv("lifu_markers_1_2back_active.csv")
calibration_complete =markers_df[markers_df["marker"] == "START_EXPERIMENT_RECEIVED"].iloc[-1]
calibration_complete = calibration_complete["LSL_timestamp"]
markers_df_on = markers_df[markers_df["marker"] =="LIFU_ON"]
markers_df

,Time,marker,LSL_timestamp
0,2.686235,collecting_baseline,1.002150e+06
1,2.723118,collecting_baseline,1.002150e+06
2,2.756129,collecting_baseline,1.002150e+06
3,2.799655,collecting_baseline,1.002150e+06
4,2.829432,collecting_baseline,1.002150e+06
...,...,...,...
201,131.477770,START_EXPERIMENT_RECEIVED,1.002279e+06
202,131.819562,LIFU_ON,1.002279e+06
203,136.830203,LIFU_OFF,1.002284e+06
204,358.441138,LIFU_ON,1.002506e+06


In [4]:
calibration_complete  # remove w actual marker time

1002278.9429925

In [6]:
eeg_df = pd.read_csv("scope_eeg_2back_active.csv")
eeg_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06,Ch07
0,1.002148e+06,36667.449219,2295.493408,5.269290e+06,7.997383e+06,1.913249e+06,1.300813e+06,-76.059860
1,1.002148e+06,36902.093750,1443.515015,2.083736e+06,8.034562e+06,1.922143e+06,1.921479e+06,-75.114929
2,1.002148e+06,37381.917969,537.728394,2.891518e+05,8.034830e+06,1.922208e+06,1.921479e+06,-74.131935
3,1.002148e+06,37252.261719,-402.884125,1.623156e+05,8.031784e+06,1.921479e+06,1.921479e+06,-72.873756
4,1.002148e+06,36728.929688,-1358.252930,1.844851e+06,8.062043e+06,1.928718e+06,1.921479e+06,-71.237610
...,...,...,...,...,...,...,...,...
93475,1.002522e+06,37745.835938,-1.332321,1.775080e+00,9.415363e-01,-3.297760e-01,-3.055519e-01,0.004536
93476,1.002522e+06,37685.875000,-1.410046,1.988230e+00,9.016770e-01,-3.393117e-01,-3.055519e-01,0.174408
93477,1.002522e+06,38178.898438,-1.447061,2.093984e+00,8.713247e-01,-3.465730e-01,-3.055519e-01,0.248722
93478,1.002523e+06,0.000000,-1.441624,2.078281e+00,8.508972e-01,-3.514600e-01,-3.055519e-01,-1.428712


In [7]:
eeg_df = eeg_df.rename(columns = {"Time": "LSL Time", "Ch01": "Raw", "Ch02": "Theta Bandpass", "Ch03": "Power", "Ch04": "Moving Average", "Ch05": "Z-score", "Ch06": "Hold", "Ch07": "Trigger Channel"})
eeg_df

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel
0,1.002148e+06,36667.449219,2295.493408,5.269290e+06,7.997383e+06,1.913249e+06,1.300813e+06,-76.059860
1,1.002148e+06,36902.093750,1443.515015,2.083736e+06,8.034562e+06,1.922143e+06,1.921479e+06,-75.114929
2,1.002148e+06,37381.917969,537.728394,2.891518e+05,8.034830e+06,1.922208e+06,1.921479e+06,-74.131935
3,1.002148e+06,37252.261719,-402.884125,1.623156e+05,8.031784e+06,1.921479e+06,1.921479e+06,-72.873756
4,1.002148e+06,36728.929688,-1358.252930,1.844851e+06,8.062043e+06,1.928718e+06,1.921479e+06,-71.237610
...,...,...,...,...,...,...,...,...
93475,1.002522e+06,37745.835938,-1.332321,1.775080e+00,9.415363e-01,-3.297760e-01,-3.055519e-01,0.004536
93476,1.002522e+06,37685.875000,-1.410046,1.988230e+00,9.016770e-01,-3.393117e-01,-3.055519e-01,0.174408
93477,1.002522e+06,38178.898438,-1.447061,2.093984e+00,8.713247e-01,-3.465730e-01,-3.055519e-01,0.248722
93478,1.002523e+06,0.000000,-1.441624,2.078281e+00,8.508972e-01,-3.514600e-01,-3.055519e-01,-1.428712


In [8]:
lifu_on = markers_df[markers_df["marker"]=="LIFU_ON"].copy()
lifu_on_timestamp = lifu_on["LSL_timestamp"]
lifu_on_timestamp = np.array(lifu_on_timestamp)
lifu_on_timestamp

array([1002279.2847846, 1002505.9063606])

In [9]:
target_ts = lifu_on_timestamp

tol = 3e-3  # tolerance for float comparison

df_matches = eeg_df[
    eeg_df["LSL Time"].apply(
        lambda x: np.any(np.abs(target_ts - x) < tol)
    )
]

df_matches

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel
32754,1.002279e+06,36911.042969,1.092012,1.192490,8.509987,1.480858,1.504728,0.285806
32755,1.002279e+06,37432.792969,0.989552,0.979214,8.473432,1.472113,1.504728,0.245151
89403,1.002506e+06,38323.687500,-3.652433,13.340266,8.865489,1.565907,1.565907,0.942968
89404,1.002506e+06,38446.574219,-3.700580,13.694290,8.902260,1.574703,1.565907,0.758015


In [10]:
merged_df = eeg_df

In [11]:
buffer = []
offline_z = []
medians = []
mads = []
last_theta_val = None
last_median_val = None
last_mad_val = None

SONICATION_TIME = 5 #seconds i believe  
COOLDOWN_TIME = 15 #sonication time + cooldown time 
THETA_THRESHOLD_Z = 1.5    # z-score threshold
MU = 3.12
SIGMA =  5.31
MAD_THRESHOLD = 6       # for artifact rejection in baseline collection
INITIAL_CUTOFF = 100.0   # initial power threshold to exclude extreme artifacts
BUFFER_SIZE = 500
sonication_enabled = calibration_complete

# Ch05 = index 4
channel = "Hold"

for i in range(len(merged_df)):
    theta_val = merged_df[channel].iloc[i]   # MUST match online
    if last_theta_val is not None and theta_val == last_theta_val:
        offline_z.append(last_theta_val)
        medians.append(last_median_val)
        mads.append(last_theta_val)
        continue
    # Warm-up
    if len(buffer) <= 200:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        offline_z.append(np.nan)
        medians.append(np.nan)
        mads.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6
    

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        offline_z.append(np.nan)
        medians.append(np.nan)
        mads.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)
    last_theta_val = theta_val
    last_median_val = median
    last_mad_val = mad

    # Z-score
    offline_z.append(theta_val)
    medians.append(median)
    mads.append(mad)

merged_df["offline z-score"] = offline_z
merged_df["median"] = medians
merged_df['mad'] = mads




In [12]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["offline z-score"].iloc[i]
    now = merged_df["LSL Time"].iloc[i]

    if now> sonication_enabled and theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD and (now - last_trigger_time > COOLDOWN_TIME):
        LIFU[i] = 1.0
        last_trigger_time = now


In [13]:
merged_df["LIFU"] = LIFU
merged_df

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU
0,1.002148e+06,36667.449219,2295.493408,5.269290e+06,7.997383e+06,1.913249e+06,1.300813e+06,-76.059860,NaN,NaN,NaN,0.0
1,1.002148e+06,36902.093750,1443.515015,2.083736e+06,8.034562e+06,1.922143e+06,1.921479e+06,-75.114929,NaN,NaN,NaN,0.0
2,1.002148e+06,37381.917969,537.728394,2.891518e+05,8.034830e+06,1.922208e+06,1.921479e+06,-74.131935,NaN,NaN,NaN,0.0
3,1.002148e+06,37252.261719,-402.884125,1.623156e+05,8.031784e+06,1.921479e+06,1.921479e+06,-72.873756,NaN,NaN,NaN,0.0
4,1.002148e+06,36728.929688,-1358.252930,1.844851e+06,8.062043e+06,1.928718e+06,1.921479e+06,-71.237610,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
93475,1.002522e+06,37745.835938,-1.332321,1.775080e+00,9.415363e-01,-3.297760e-01,-3.055519e-01,0.004536,-0.305552,-0.047599,-0.305552,0.0
93476,1.002522e+06,37685.875000,-1.410046,1.988230e+00,9.016770e-01,-3.393117e-01,-3.055519e-01,0.174408,-0.305552,-0.047599,-0.305552,0.0
93477,1.002522e+06,38178.898438,-1.447061,2.093984e+00,8.713247e-01,-3.465730e-01,-3.055519e-01,0.248722,-0.305552,-0.047599,-0.305552,0.0
93478,1.002523e+06,0.000000,-1.441624,2.078281e+00,8.508972e-01,-3.514600e-01,-3.055519e-01,-1.428712,-0.305552,-0.047599,-0.305552,0.0


In [14]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on_df = merged_df[mask_lifu]
lifu_on_df["LSL Time"] = lifu_on_df["LSL Time"]
lifu_on_cal = lifu_on_df[lifu_on_df["LSL Time"] > calibration_complete]
lifu_on_cal

C:\Users\jshin\AppData\Local\Temp\ipykernel_16088\4113107194.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lifu_on_df["LSL Time"] = lifu_on_df["LSL Time"]


,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU
32753,1.002279e+06,36895.179688,1.126410,1.268799,8.609761,1.504728,1.504728,0.322021,1.504728,0.078045,0.412913,1.0
89402,1.002506e+06,37761.457031,-3.544332,12.562289,8.799663,1.550159,1.565907,0.871264,1.565907,-0.067164,0.317587,1.0


In [15]:
lifu_on_time = lifu_on_cal["LSL Time"]
lifu_on_time = np.array(lifu_on_time)
offline_lifu= lifu_on_time

In [16]:
offline_lifu # because the begin calibration didn't start yet that's why it doesn't

array([1002279.2816333, 1002505.9032047])

In [17]:
target_ts

array([1002279.2847846, 1002505.9063606])

In [18]:
calibration_complete

1002278.9429925

In [19]:
offline_lifu - target_ts

array([-0.0031513, -0.0031559])

In [26]:
channel_8 = eeg_df[eeg_df['LSL Time'] > calibration_complete]
channel_8 = channel_8.nlargest(25, "Trigger Channel")
channel_8

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU
89671,1.002507e+06,38284.136719,-0.271589,0.073760,1.264270,-0.252567,-0.078539,539.357605,-0.078539,-0.093485,-0.078539,0.0
89621,1.002507e+06,38287.429688,-0.862595,0.744070,4.315446,0.477380,0.317375,537.825012,0.317375,-0.093485,0.317375,0.0
89871,1.002508e+06,38243.582031,-0.334412,0.111832,0.667704,-0.395286,-0.402801,532.782104,-0.402801,-0.116406,0.283838,0.0
90671,1.002511e+06,38158.800781,0.560987,0.314706,1.333185,-0.236080,-0.191090,532.646179,-0.191090,-0.164604,-0.191090,0.0
89721,1.002507e+06,38281.542969,-0.338397,0.114513,4.407490,0.499399,0.377042,532.595520,0.377042,-0.108489,0.377042,0.0
90721,1.002511e+06,38145.492188,1.935161,3.744846,2.308223,-0.002817,-0.176350,532.311218,-0.176350,-0.164604,-0.176350,0.0
89921,1.002508e+06,38238.523438,1.488304,2.215049,1.237758,-0.258909,-0.269532,532.294739,-0.269532,-0.138915,-0.269532,0.0
90171,1.002509e+06,38212.449219,0.044028,0.001938,1.133909,-0.283754,-0.298228,532.269836,-0.298228,-0.160214,-0.298228,0.0
90521,1.002510e+06,38175.894531,-0.583492,0.340463,2.888506,0.136006,0.230066,532.190308,0.230066,-0.161698,0.230066,0.0
90621,1.002511e+06,38159.656250,-1.543460,2.382268,0.867365,-0.347520,-0.391349,532.186890,-0.391349,-0.164604,-0.391349,0.0


In [28]:
# channel_8 = eeg_df.nlargest(15, "Trigger Channel")
channel_8_sorted = channel_8.sort_values("LSL Time", ascending=False) # descending
MIN_SPACING = 10.0  # seconds
kept_rows = []
last_time = 0

for _, row in channel_8_sorted.iterrows():
    t = row["LSL Time"]
    if np.abs(t - last_time) >= MIN_SPACING:
        kept_rows.append(row)
        last_time = t
    if len(kept_rows) == 15:
        break
ttl_signal = pd.DataFrame(kept_rows)
ttl_signal_time = np.array(ttl_signal['LSL Time'])
ttl_signal_time = np.sort(ttl_signal_time)
target_ts- ttl_signal_time

array([-1.0603386, -5.2712901])

In [22]:
#windowing raw eeg
raw_with08 = pd.read_csv("offline_2back_sham.csv")
raw = raw_with08.drop(' Ch08', axis = 1)
pre = 2
post = 10
raw_windows = []
target_ts_onset = np.array(lifu_on['Time'])


for idx, event in enumerate(target_ts_onset): 
    mask = (raw['Time'] >= (event - pre)) & (raw['Time'] <= (event + post))
    window_df = raw.loc[mask].copy()
    window_df["t_rel"] = window_df["Time"] - event 
    window_df["idx"] = idx
    raw_windows.append(window_df)
raw_windows[0]

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06,Ch07,t_rel,idx
32455,129.820,-16270.854492,-56878.667969,-66580.250000,30797.009766,-60796.386719,-15525.200195,-15654.398438,-1.999562,0
32456,129.824,-16494.181641,-56393.367188,-66324.312500,30707.244141,-60248.093750,-15600.373047,-15968.490234,-1.995562,0
32457,129.828,-16609.312500,-57270.335938,-66829.601562,30812.052734,-61362.648438,-15531.349609,-14981.748047,-1.991562,0
32458,129.832,-16391.302734,-57910.355469,-67176.726562,30917.574219,-62114.105469,-15449.357422,-14512.921875,-1.987562,0
32459,129.836,-16272.451172,-57079.054688,-66696.312500,30810.287109,-61041.093750,-15513.112305,-15480.734375,-1.983562,0
...,...,...,...,...,...,...,...,...,...,...
35450,141.800,-27594.568359,-69072.875000,-78300.500000,19178.390625,-73072.273438,-26638.484375,-25673.869141,9.980438,0
35451,141.804,-27456.931641,-68334.312500,-77874.960938,19087.339844,-72109.539062,-26688.458984,-26576.640625,9.984438,0
35452,141.808,-27630.234375,-67514.750000,-77419.562500,18957.880859,-71125.085938,-26791.884766,-27281.072266,9.988438,0
35453,141.812,-27798.843750,-68075.648438,-77760.726562,19013.599609,-71863.921875,-26762.943359,-26553.464844,9.992438,0


In [29]:
#windowing scope eeg
pre = 2
post = 7
theta_windows = []


for idx, event in enumerate(target_ts): 
    mask = (eeg_df['LSL Time'] >= (event - pre)) & (eeg_df['LSL Time'] <= (event + post))
    window_df = eeg_df.loc[mask].copy()
    window_df["t_rel"] = window_df["LSL Time"] - event 
    window_df["idx"] = idx
    theta_windows.append(window_df)
theta_windows[0]

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU,t_rel,idx
32257,1.002277e+06,36958.769531,-1.139847,1.299251,2.421673,0.024324,0.027169,-0.136723,0.027169,0.126161,0.027169,0.0,-1.999572,0
32258,1.002277e+06,36780.144531,-1.113436,1.239738,2.380751,0.014534,0.027169,-0.315151,0.027169,0.126161,0.027169,0.0,-1.984766,0
32259,1.002277e+06,37269.906250,-1.057071,1.117400,2.320248,0.000059,0.027169,-0.440868,0.027169,0.126161,0.027169,0.0,-1.980783,0
32260,1.002277e+06,37514.593750,-0.973277,0.947269,2.240512,-0.019016,0.027169,-0.449122,0.027169,0.126161,0.027169,0.0,-1.978232,0
32261,1.002277e+06,37067.894531,-0.865270,0.748692,2.143462,-0.042234,-0.097605,-0.353430,-0.097605,0.126161,0.449418,0.0,-1.976765,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34498,1.002286e+06,37371.992188,-0.840298,0.706100,1.272161,-0.250679,-0.265979,-0.972125,-0.265979,0.442867,-0.265979,0.0,6.984868,0
34499,1.002286e+06,36851.312500,-0.781069,0.610069,1.267695,-0.251748,-0.265979,-0.886099,-0.265979,0.442867,-0.265979,0.0,6.985573,0
34500,1.002286e+06,37195.949219,-0.720331,0.518877,1.256296,-0.254475,-0.265979,-0.736612,-0.265979,0.442867,-0.265979,0.0,6.993094,0
34501,1.002286e+06,37682.917969,-0.658114,0.433114,1.237830,-0.258892,-0.265979,-0.533921,-0.265979,0.442867,-0.265979,0.0,6.994558,0


In [31]:
#base_dir_raw = r"C:\Users\jshin\OW_closedloopLIFU\eeg_windows\raw"
base_dir_scope=r"C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope"
# MAKE SURE TO PUT IN NAME
new_folder_name = "active"
#new_folder_path_raw = os.path.join(base_dir_raw, new_folder_name)
new_folder_path_scope = os.path.join(base_dir_scope, new_folder_name)
#os.makedirs(new_folder_path_raw, exist_ok=False) # check
os.makedirs(new_folder_path_scope, exist_ok=False) # check

for idx, event in enumerate(target_ts_onset):
    # Build a unique filename for each event
    #output_path_raw = os.path.join(new_folder_path_raw, f"{idx}_{event}_eeg_raw.csv")
    output_path_scope = os.path.join(new_folder_path_scope, f"{idx}_{event}_eeg_scope.csv")
    try:
        #raw_windows[idx].to_csv(output_path_raw, index=False, encoding="utf-8")
        theta_windows[idx].to_csv(output_path_scope, index=False, encoding="utf-8")
        #print(f"Saved successfully to: {output_path_raw}")
        print(f"Saved successfully to: {output_path_scope}")
    except (OSError, IOError) as e:
        print(f"Error saving CSV file: {e}")


Saved successfully to: C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope\active\0_131.81956199998967_eeg_scope.csv
Saved successfully to: C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope\active\1_358.4411379999947_eeg_scope.csv
